In [ ]:
#| default_exp latechunk_eval
#| eval: false

# Embedding-model benchmark: accuracy & speed

Benchmarks embedding models on **LongEmbed** retrieval tasks, comparing chunking / retrieval
strategies on two axes: **accuracy** (nDCG@10, Recall@{1,5,10}, MRR@10) and **speed**
(index-build encode time, per-query latency).

What's covered:

| model | strategy | notes |
|---|---|---|
| `potion-retrieval-32M` (static) | naive `+` rerank | fast baseline |
| `bge-micro-v2` | naive `+` rerank | small bi-encoder baseline |
| `jina-embeddings-v2-small-en` | naive, **late-chunk** `×` **matryoshka dims** (512/384/256/128) | 8192-token context, long-chunk strategy |
| `nomic-embed-text-v1.5` | late-chunk | long-context reference point |
| **ColBERT** route | late interaction (MaxSim) | multi-vector, brute-force scored |

**Everything is a reusable function** and **every expensive step is cached** — datasets, encoders,
per-model DBs (one ANN store per strategy/dim) and result rows all persist under `bench/`.
Re-running a cell skips anything already built, so you never recompute what you don't have to.
Run one model's cell in isolation, or the whole leaderboard at the end.

> Requires the eval extras: `uv sync --extra eval` (or `pip install litesearch[eval]`).

## 0 · Setup

In [ ]:
#| eval: false
import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())
import requests as _r, urllib3 as _u
_u.disable_warnings()
_orig_send = _r.Session.send
_r.Session.send = lambda self, *a, **k: _orig_send(self, *a, **{**k, 'verify': False})

import json, os, time
from dataclasses import dataclass
import numpy as np, pandas as pd
from datasets import load_dataset
from fastcore.all import AttrDict, L
from litesearch import *
from litesearch.data import chunk_spans
from litesearch.utils import FastEncode, AutoLateChunkFastEncode, LateChunkFastEncode, static_retrieval_embedder, nomic_text_v15, bge_model
from chonkie import FastChunker

## 1 · Datasets (cached)

`load_longembed` pulls a LongEmbed task, caps the corpus/queries, and keeps only queries whose
judged document survives the cap. The `Benchmark` object caches the loaded corpus/queries/qrels to
`bench/data_<task>.json`, so the dataset is pulled from HF at most once.

In [ ]:
#| eval: false
TASKS = ['narrativeqa', '2wikimqa']

def load_longembed(task, max_docs=200, max_queries=100):
    'Load a LongEmbed task capped to max_docs/max_queries; keep only queries whose judged doc survives the cap.'
    ds = load_dataset('dwzhu/LongEmbed', task)
    corpus = {r['doc_id']: r['text'] for r in ds['corpus'].select(range(min(max_docs, len(ds['corpus']))))}
    queries = {r['qid']: r['text'] for r in ds['queries']}
    qrels = {}
    for r in ds['qrels']:
        qid, did = r['qid'], r['doc_id']
        if did in corpus and qid in queries: qrels.setdefault(qid, set()).add(did)
    queries = {q: queries[q] for q in list(qrels)[:max_queries]}
    qrels = {q: qrels[q] for q in queries}
    return corpus, queries, qrels

## 2 · Metrics (pure functions)

In [ ]:
#| eval: false
def agg_docs(hits):
    'Collapse ranked chunk hits to unique parent doc_ids, keeping best rank.'
    seen, out = set(), []
    for h in hits:
        d = json.loads(h['metadata'])['doc_id']
        if d not in seen: seen.add(d); out.append(d)
    return out

def recall_at_k(ranked, relevant, k):
    'Fraction of relevant docs present in the top-k ranking.'
    if not relevant: return 0.0
    return len(set(ranked[:k]) & relevant) / len(relevant)

def ndcg_at_k(ranked, relevant, k=10):
    'Binary-relevance nDCG@k.'
    dcg = sum(1.0/np.log2(i+2) for i,d in enumerate(ranked[:k]) if d in relevant)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), k)))
    return float(dcg/idcg) if idcg else 0.0

def mrr_at_k(ranked, relevant, k=10):
    'Reciprocal rank of the first relevant doc within top-k (0 if none).'
    for i,d in enumerate(ranked[:k]):
        if d in relevant: return 1.0/(i+1)
    return 0.0

In [ ]:
#| eval: false
_hits = [{'metadata': json.dumps({'doc_id':'A'})}, {'metadata': json.dumps({'doc_id':'B'})},
         {'metadata': json.dumps({'doc_id':'A'})}]
assert agg_docs(_hits) == ['A','B']
assert recall_at_k(['A','B','C'], {'A','C'}, 3) == 1.0
assert recall_at_k(['A','B','C'], {'A','C'}, 1) == 0.5
assert abs(ndcg_at_k(['A','B'], {'A'}, 10) - 1.0) < 1e-9
assert ndcg_at_k(['B','A'], {'A'}, 10) < 1.0
assert mrr_at_k(['B','A'], {'A'}, 10) == 0.5 and mrr_at_k(['B','C'], {'A'}, 10) == 0.0

## 3 · Chunking strategies

A strategy is just a `chonkie` chunker factory. Long-context models (Jina, Nomic) get the `long`
strategy — bigger chunks that (a) cut the vector count and (b) let late-chunking pool over more of
the document per forward pass, actually exercising the 8192-token window. ColBERT gets `small`
passage-sized chunks so MaxSim stays discriminative.

In [ ]:
#| eval: false
CHUNKERS = {
    'default': lambda: FastChunker(),                    # ~4096 chars, general purpose
    'long':    lambda: FastChunker(chunk_size=8000),     # long-context models: fewer, richer chunks
    'small':   lambda: FastChunker(chunk_size=1500),     # ColBERT passage-sized chunks
}
def make_chunker(key): return CHUNKERS[key]()

## 4 · Encoders & model registry

One encoder is built **per model and reused** across every method and matryoshka dim. Kinds:

- **static** (`model2vec`) / **fast** (`FastEncode`) → bi-encoders, used for `naive`.
- **late** (`AutoLateChunkFastEncode`) → a `FastEncode` subclass, so the *same* object does `naive`
  (pooled) **and** `latechunk` (embed whole doc, pool per span; auto-windowed for long docs).
- **colbert** (`ColbertEncode` below) → emits per-token vectors for late interaction.

**Matryoshka:** Jina v2 emits 512-dim vectors. We benchmark truncate-and-renormalize at 384/256/128
(`_truncate`) as a speed/size knob. The document is **encoded once at full dim**; each dim variant is
a cheap slice of the same vectors (see `Benchmark.float_rows`). Note v2-small is *not* MRL-trained, so
truncation is lossy — the table quantifies exactly how much. (For true Matryoshka use jina-v3.)

In [ ]:
#| eval: false
# Jina v2 small: JinaBERT + ALiBi, 8192-token context, mean-pooled, 512-dim, plain text (no instruction prefix).
jina_small = AttrDict(model='Xenova/jina-embeddings-v2-small-en', onnx_path='onnx/model.onnx',
                      prompt=AttrDict(document='{text}', query='{text}'))

class ColbertEncode(LateChunkFastEncode):
    'Token-level (multi-vector) encoder for ColBERT-style late interaction. Emits L2-normalized per-token vectors.'
    def encode_tokens(self, texts, prompt=None):
        out = []
        for t in L(texts):
            token_embs, _offsets, msk = self._token_embeddings(t)      # msk zeros special/pad tokens
            idx = np.where(np.asarray(msk) > 0)[0]
            v = np.asarray(token_embs)[idx].astype(np.float32)
            if v.shape[0] == 0: v = np.asarray(token_embs)[:1].astype(np.float32)
            v = v / np.clip(np.linalg.norm(v, axis=1, keepdims=True), 1e-12, None)
            out.append(v.astype(self.dtype))
        return out

def _enc_docs(enc, texts):
    'Encode documents, preferring encode_document (FastEncode) then falling back to encode (StaticModel).'
    fn = getattr(enc, 'encode_document', None) or enc.encode
    return np.asarray(fn(list(texts)), dtype=np.float32)

def _enc_query(enc, text):
    fn = getattr(enc, 'encode_query', None) or enc.encode
    return np.asarray(fn([text]), dtype=np.float32)[0]

def _truncate(e, dim):
    'Matryoshka: keep the first `dim` dims and renormalize (no-op if dim is None/>=len).'
    e = np.asarray(e, dtype=np.float32)
    if dim and dim < e.shape[-1]:
        e = e[..., :dim]
        e = e / np.clip(np.linalg.norm(e, axis=-1, keepdims=True), 1e-12, None)
    return e

In [ ]:
#| eval: false
# kind: static|fast|late|colbert  •  methods: default strategies  •  dims: matryoshka dims (None = full)
# ColBERT defaults to late interaction over Jina's token embeddings (guaranteed runnable, apples-to-apples
# vs jina-latechunk). To benchmark a purpose-trained ColBERT, point ctor at its ONNX, e.g.
#   AttrDict(model='answerdotai/answerai-colbert-small-v1', onnx_path='onnx/model.onnx', prompt=AttrDict(document='{text}',query='{text}'))
EMBEDDERS = {
    'potion':  dict(kind='static', ctor=lambda: static_retrieval_embedder(),                       methods=['naive'],              dims=[None], chunk='default'),
    'bge':     dict(kind='fast',   ctor=lambda: FastEncode(bge_model),                              methods=['naive'],              dims=[None], chunk='default'),
    'jina':    dict(kind='late',   ctor=lambda: AutoLateChunkFastEncode(model_dict=jina_small, max_seq_len=8192),
                                                                                                    methods=['naive','latechunk'],  dims=[None,384,256,128], chunk='long'),
    'nomic':   dict(kind='late',   ctor=lambda: AutoLateChunkFastEncode(model_dict=nomic_text_v15, max_seq_len=8192),
                                                                                                    methods=['latechunk'],          dims=[None], chunk='long'),
    'colbert': dict(kind='colbert',ctor=lambda: ColbertEncode(model_dict=jina_small, max_seq_len=512), methods=['colbert'],         dims=[None], chunk='small'),
}

## 5 · Row builders & single-vector storage

Builders return **float rows** (full-dim `np.float32` embeddings). `finalize_rows` then applies the
matryoshka truncation and casts to fp16 bytes — so the expensive encode happens once and every dim
variant is a cheap re-slice. `store_rows` inserts into an ANN (HNSW) store and builds the index; we
search **through the ANN index** (unlike the old flow, which built indexes but never used them).

In [ ]:
#| eval: false
def _frow(content, did, ci, emb):
    return dict(content=content, metadata=json.dumps({'doc_id':did,'chunk_idx':ci}), emb=np.asarray(emb, dtype=np.float32))

def frows_naive(corpus, enc, chunker_key):
    'Chunk then embed each chunk independently.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt, make_chunker(chunker_key))
        if not spans: continue
        embs = _enc_docs(enc, [t for _,_,t in spans])
        rows += [_frow(t,did,ci,e) for ci,((_,_,t),e) in enumerate(zip(spans,embs))]
    return rows

def frows_fulldoc(corpus, enc, chunker_key=None):
    'One embedding per whole document.'
    dids = list(corpus); embs = _enc_docs(enc, [corpus[d] for d in dids])
    return [_frow(corpus[d], d, 0, e) for d,e in zip(dids, embs)]

def frows_latechunk(corpus, lc, chunker_key):
    'Late chunking: embed the whole doc (auto-windowed if long), then mean-pool per chunk span.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt, make_chunker(chunker_key))
        if not spans: continue
        embs,_ = lc.encode_auto(txt, [(s,e) for s,e,_ in spans])
        rows += [_frow(t,did,ci,e) for ci,((_,_,t),e) in enumerate(zip(spans,embs))]
    return rows

FROWS = {'naive':frows_naive, 'fulldoc':frows_fulldoc, 'latechunk':frows_latechunk}

def finalize_rows(frows, dim=None):
    'Truncate (matryoshka) + cast to fp16 bytes for storage. Cheap: reuses already-computed embeddings.'
    out = []
    for r in frows:
        e = _truncate(r['emb'], dim)
        out.append(dict(content=r['content'], metadata=r['metadata'], embedding=e.astype(np.float16).tobytes()))
    return out

def store_rows(byte_rows, db, name):
    'Insert fp16 rows into an ANN store and build its HNSW index. ndim inferred from the first row.'
    ndim = len(np.frombuffer(byte_rows[0]['embedding'], dtype=np.float16))
    st = db.get_store(name=name, ann=True, ndim=ndim, metric='cosine')
    st.insert_all(byte_rows); st.rebuild_index()
    return st

def store_count(db, name):
    'Row count of a store (0 if it does not exist yet) — used to skip already-built stores.'
    try: return db.t[name].count if name in db.t else 0
    except Exception: return 0

## 6 · ColBERT late-interaction route

ColBERT keeps a vector per token and scores a (query, chunk) pair by **MaxSim** — for each query
token, the best-matching doc token, summed. Chunks are stored as token matrices in a plain table and
scored brute-force, which is fine at benchmark corpus sizes. This is a genuinely different retrieval
path from single-vector ANN, so it gets its own store + evaluator.

In [ ]:
#| eval: false
def maxsim(q_tok, d_tok):
    'ColBERT MaxSim: sum over query tokens of the max dot-product against any doc token (rows are L2-normalized).'
    if q_tok.shape[0]==0 or d_tok.shape[0]==0: return 0.0
    return float((q_tok @ d_tok.T).max(axis=1).sum())

def frows_colbert(corpus, cb, chunker_key):
    'Encode each chunk to a token matrix (multi-vector) for late interaction.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt, make_chunker(chunker_key))
        if not spans: continue
        mats = cb.encode_tokens([t for _,_,t in spans])
        for ci,((_,_,t),m) in enumerate(zip(spans,mats)):
            m = np.asarray(m, dtype=np.float16)
            rows.append(dict(content=t, metadata=json.dumps({'doc_id':did,'chunk_idx':ci}),
                             tokens=m.tobytes(), ntok=m.shape[0], dim=m.shape[1]))
    return rows

def colbert_store(byte_rows, db, name):
    'Store multi-vector chunks (token-matrix bytes + shape) in a plain, non-ANN table.'
    if name in db.t: db.t[name].drop()
    t = db.t[name].create(content=str, metadata=str, tokens=bytes, ntok=int, dim=int, id=int, pk='id')
    t.insert_all(byte_rows)
    return t

def evaluate_colbert(db, table, queries, qrels, cb, reranking=False, rerank_model=None):
    'Brute-force MaxSim ranking over all chunks; returns accuracy metrics + mean query latency (ms).'
    rows = list(db.t[table]())
    mats = [np.frombuffer(r['tokens'], dtype=np.float16).reshape(r['ntok'], r['dim']).astype(np.float32) for r in rows]
    metas = [r['metadata'] for r in rows]; texts = [r['content'] for r in rows]
    ndcg=r1=r5=r10=mrr=0.0; lat=[]; n=len(queries)
    for qid,qt in queries.items():
        qm = np.asarray(cb.encode_tokens([qt])[0], dtype=np.float32)
        t0 = time.perf_counter()
        order = np.argsort([maxsim(qm, m) for m in mats])[::-1][:100]
        hits = [{'metadata':metas[i], 'content':texts[i]} for i in order]
        if reranking: hits = rerank_hits(qt, hits, rerank_model)
        lat.append(time.perf_counter()-t0)
        ranked = agg_docs(hits); rel = qrels[qid]
        ndcg += ndcg_at_k(ranked, rel, 10); mrr += mrr_at_k(ranked, rel, 10)
        r1 += recall_at_k(ranked, rel, 1); r5 += recall_at_k(ranked, rel, 5); r10 += recall_at_k(ranked, rel, 10)
    return {'ndcg@10':ndcg/n, 'recall@1':r1/n, 'recall@5':r5/n, 'recall@10':r10/n,
            'mrr@10':mrr/n, 'query_ms':1000*float(np.mean(lat))}

In [ ]:
#| eval: false
# MaxSim sanity: an exact-token-overlap doc must outscore an unrelated one.
_q = np.eye(3, dtype=np.float32)[:2]                 # two orthonormal query "tokens"
_match = np.eye(3, dtype=np.float32)                 # contains both
_other = np.eye(3, dtype=np.float32)[2:]             # contains neither
assert maxsim(_q, _match) > maxsim(_q, _other)
assert abs(maxsim(_q, _match) - 2.0) < 1e-6
assert maxsim(_q, np.zeros((0,3), dtype=np.float32)) == 0.0

## 7 · Single-vector evaluation (accuracy + speed)

In [ ]:
#| eval: false
def evaluate(db, table, queries, qrels, enc, dim=None, reranking=False, rerank_model=None):
    'Query-averaged nDCG@10, Recall@{1,5,10}, MRR@10 and mean query latency (ms) over one ANN store.'
    ndcg=r1=r5=r10=mrr=0.0; lat=[]; n=len(queries)
    for qid,qt in queries.items():
        qe = _truncate(_enc_query(enc, qt), dim).astype(np.float16).tobytes()
        t0 = time.perf_counter()
        hits = db.search(qt, qe, columns=['metadata'], limit=100, table_name=table, ann=True,
                         reranking=reranking, rerank_model=rerank_model)
        lat.append(time.perf_counter()-t0)
        ranked = agg_docs(hits); rel = qrels[qid]
        ndcg += ndcg_at_k(ranked, rel, 10); mrr += mrr_at_k(ranked, rel, 10)
        r1 += recall_at_k(ranked, rel, 1); r5 += recall_at_k(ranked, rel, 5); r10 += recall_at_k(ranked, rel, 10)
    return {'ndcg@10':ndcg/n, 'recall@1':r1/n, 'recall@5':r5/n, 'recall@10':r10/n,
            'mrr@10':mrr/n, 'query_ms':1000*float(np.mean(lat))}

## 8 · Smart benchmark orchestrator

A `Run` is `(model, method, dim, chunk, rerank)`. Its `store()` name identifies the vectors on disk
(shared by base + reranked variants — reranking is query-time only). `Benchmark` memoizes datasets,
encoders, DBs and full-dim float rows, and caches every result row to `bench/results.json`. So it:

- builds each store **at most once** (skips if it already has rows),
- encodes each `(model, method, chunk)` **once**, then slices cheap matryoshka dim variants,
- returns cached metrics instantly on re-run.

Pass `force_build=True` / `force_eval=True` to override, or run a subset with `expand_runs(models=[...])`.

In [ ]:
#| eval: false
def _slug(s): return ''.join(c if c.isalnum() else '_' for c in s)

@dataclass
class Run:
    model:  str
    method: str
    dim:    int = None
    chunk:  str = 'default'
    rerank: bool = False
    def base_key(self):  return f'{self.model}|{self.method}|{self.chunk}'            # -> shared full-dim float rows
    def store(self):     return f'{self.method}__{self.chunk}__d{self.dim or "full"}' # -> per-dim vectors on disk
    def label(self):
        d = '' if self.dim is None else f'-{self.dim}'
        return f'{self.model}{d}-{self.method}' + ('+rr' if self.rerank else '')

def expand_runs(models=None, add_rerank=True):
    'Default Run list from the registry: one per (model, method, dim), plus a reranked variant.'
    runs = []
    for m in (models or EMBEDDERS):
        spec = EMBEDDERS[m]
        for method in spec['methods']:
            for dim in spec.get('dims',[None]):
                runs.append(Run(m, method, dim=dim, chunk=spec.get('chunk','default')))
                if add_rerank: runs.append(Run(m, method, dim=dim, chunk=spec.get('chunk','default'), rerank=True))
    return runs

In [ ]:
#| eval: false
class Benchmark:
    'Caches datasets, encoders, per-model DBs and result rows so nothing is recomputed unnecessarily.'
    METRICS = ['ndcg@10','recall@1','recall@5','recall@10','mrr@10','query_ms']
    # sem_search=False: we search via the usearch *Python* HNSW index (ann=True) + native FTS5, so the
    # usearch SQLite extension (a GitHub-hosted binary) is never needed — one less download to depend on.
    def __init__(self, tasks=TASKS, max_docs=100, max_queries=50, cache='bench', loader=None, sem_search=False):
        self.tasks, self.loader = tasks, loader or load_longembed
        self.max_docs, self.max_queries, self.cache, self.sem_search = max_docs, max_queries, cache, sem_search
        os.makedirs(cache, exist_ok=True)
        self._enc, self._data, self._db, self._frows = {}, {}, {}, {}
        p = f'{cache}/results.json'; self.results = json.load(open(p)) if os.path.exists(p) else {}
    def _save(self): json.dump(self.results, open(f'{self.cache}/results.json','w'), indent=1)

    def data(self, task):
        'Corpus/queries/qrels, memoized in RAM and on disk.'
        if task in self._data: return self._data[task]
        p = f'{self.cache}/data_{task}.json'
        if os.path.exists(p):
            d = json.load(open(p)); c,q,r = d['corpus'], d['queries'], {k:set(v) for k,v in d['qrels'].items()}
        else:
            c,q,r = self.loader(task, self.max_docs, self.max_queries)
            json.dump({'corpus':c,'queries':q,'qrels':{k:sorted(v) for k,v in r.items()}}, open(p,'w'))
        self._data[task] = (c,q,r); return self._data[task]

    def encoder(self, model):
        'Build an encoder once per model and reuse it across methods/dims.'
        if model not in self._enc: self._enc[model] = EMBEDDERS[model]['ctor']()
        return self._enc[model]

    def db(self, model, task):
        key = (model, task)
        if key not in self._db:
            self._db[key] = database(f'{self.cache}/db/{_slug(model)}/eval_{task}.db', sem_search=self.sem_search)
        return self._db[key]

    def float_rows(self, run, task, colbert=False):
        'Full-dim float rows for a (model,method,chunk); built once, reused by every dim variant. Returns (rows, encode_sec).'
        key = (task, run.base_key())
        if key in self._frows: return self._frows[key]
        corpus,_,_ = self.data(task); enc = self.encoder(run.model)
        t0 = time.perf_counter()
        rows = frows_colbert(corpus, enc, run.chunk) if colbert else FROWS[run.method](corpus, enc, run.chunk)
        self._frows[key] = (rows, time.perf_counter()-t0); return self._frows[key]

    def eval_run(self, run, task, force_build=False, force_eval=False):
        'Build (if needed) and evaluate one run on one task; cache & return its metrics dict.'
        lbl = run.label()
        cached = self.results.get(lbl, {}).get(task)
        if cached and not force_eval and not force_build: return cached
        db = self.db(run.model, task); corpus,queries,qrels = self.data(task); name = run.store()
        enc_sec = None
        if run.method == 'colbert':
            if force_build or store_count(db, name) == 0:
                rows, enc_sec = self.float_rows(run, task, colbert=True); colbert_store(rows, db, name)
            m = evaluate_colbert(db, name, queries, qrels, self.encoder(run.model), reranking=run.rerank)
        else:
            frows, enc_sec = self.float_rows(run, task)
            if force_build or store_count(db, name) == 0:
                store_rows(finalize_rows(frows, run.dim), db, name)
            m = evaluate(db, name, queries, qrels, self.encoder(run.model), dim=run.dim, reranking=run.rerank)
        m = dict(m, chunks=store_count(db,name), encode_sec=enc_sec)
        self.results.setdefault(lbl, {})[task] = m; self._save()
        return m

    def run(self, runs, force_build=False, force_eval=False):
        'Evaluate every run across every task; return a leaderboard DataFrame averaged over tasks.'
        for r in runs:
            for task in self.tasks: self.eval_run(r, task, force_build, force_eval)
        out = []
        for r in runs:
            got = [self.results[r.label()][t] for t in self.tasks if t in self.results.get(r.label(),{})]
            if not got: continue
            row = {c: float(np.mean([g[c] for g in got])) for c in self.METRICS}
            row['chunks'] = int(np.mean([g['chunks'] for g in got]))
            es = [g['encode_sec'] for g in got if g.get('encode_sec')]
            row['encode_s'] = round(float(np.mean(es)),2) if es else np.nan
            out.append(dict(run=r.label(), **row))
        df = pd.DataFrame(out).set_index('run')
        return df.round({c:4 for c in self.METRICS} | {'query_ms':2})

## 9 · Run the benchmark

Set the corpus/query caps once. Each section below is independent and cached — run just the model
you care about. Start small (`max_docs=50`) to smoke-test, then scale up.

In [ ]:
#| eval: false
bm = Benchmark(tasks=TASKS, max_docs=100, max_queries=50, cache='bench')

### 9a · Static & BGE baselines (naive + reranking)

In [ ]:
#| eval: false
bm.run(expand_runs(models=['potion','bge']))

### 9b · Jina — long context, late chunking & matryoshka

Late chunking `×` {512, 384, 256, 128} dims, each `±` reranking. `naive` is included as the pooled
baseline. Watch `query_ms` and `chunks` drop as the dim shrinks, and how much accuracy holds.

In [ ]:
#| eval: false
bm.run(expand_runs(models=['jina']))

### 9c · Nomic reference & ColBERT late interaction

In [ ]:
#| eval: false
bm.run(expand_runs(models=['nomic','colbert']))

### 9d · Combined leaderboard

All runs, sorted by nDCG@10. `encode_s` is the one-time index-build encode time per task
(shared across a model's matryoshka dims); `query_ms` is mean per-query latency; `chunks` is index size.

In [ ]:
#| eval: false
lb = bm.run(expand_runs())
lb = lb.sort_values('ndcg@10', ascending=False)
lb.to_csv('bench/leaderboard.csv')
lb

## Appendix A · Contextual retrieval (optional)

Anthropic-style contextual retrieval: prepend a short LLM-generated situating blurb to each chunk
before embedding. Needs a local LLM (`rishi` + Gemma), so it's off the default path. Register it as a
method by adding `frows_contextual` to `FROWS` and a spec to `EMBEDDERS`.

In [ ]:
#| eval: false
_SIT_CACHE = {}
def situate(chat, doc, chunk, doc_budget=2000):
    'Short doc-situating blurb via a local LLM; fresh conversation per call, non-fatal on error.'
    key = (hash(doc), hash(chunk))
    if key in _SIT_CACHE: return _SIT_CACHE[key]
    from rishi.core import resp_text
    chat.conv = chat._stack.enter_context(chat.engine.create_conversation())
    prompt = (f"<document>\n{doc[:doc_budget]}\n</document>\nHere is a chunk from it:\n<chunk>\n{chunk}\n</chunk>\n"
              "Give a short sentence situating this chunk within the document for search. Answer with only that sentence.")
    try: blurb = resp_text(chat(prompt)).strip()
    except Exception as e:
        import warnings; warnings.warn(f'situate failed ({type(e).__name__}); embedding chunk without context'); blurb = ''
    _SIT_CACHE[key] = blurb
    return blurb

def frows_contextual(corpus, enc, chunker_key, chat=None, doc_budget=2000):
    'Contextual retrieval: prepend a situating blurb to each chunk before embedding.'
    from rishi.core import Chat
    chat = chat or Chat(cache_dir='.cache/litertlm')
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt, make_chunker(chunker_key))
        texts = [f"{situate(chat, txt, t, doc_budget)}\n{t}" for _,_,t in spans]
        embs = _enc_docs(enc, texts)
        rows += [_frow(t,did,ci,e) for ci,(t,e) in enumerate(zip(texts,embs))]
    return rows

## Appendix B · Offline smoke test (mock encoders)

The framework is validated end-to-end without any network/model by swapping in deterministic mock
encoders (bag-of-hash vectors). This proves the pipeline — storage, ANN search, matryoshka
truncation, ColBERT MaxSim, caching/skip — independently of model downloads.

In [ ]:
#| eval: false
import hashlib
def _mock():
    D=256
    def h(w): return int(hashlib.md5(w.encode()).hexdigest(),16)
    def vec(t):
        v=np.zeros(D,dtype=np.float32)
        for w in t.lower().split(): v[h(w)%D]+=1.0
        return v/max(float(np.linalg.norm(v)),1e-9)
    class MF:
        def encode_document(self,l): return np.stack([vec(x) for x in l]).astype(np.float32)
        encode_query=encode_document
        def encode(self,l): return self.encode_document(l)
    class ML(MF):
        def encode_auto(self,text,spans,**k): return np.stack([vec(text[s:e]) for s,e in spans]).astype(np.float32),'normal'
    class MC:
        def encode_tokens(self,ts): return [np.stack([vec(w) for w in (t.split() or [''])]).astype(np.float32) for t in ts]
    return MF, ML, MC

def smoke_test(cache='bench_mock'):
    import shutil; shutil.rmtree(cache, ignore_errors=True)
    MF,ML,MC=_mock()
    reg=dict(EMBEDDERS)
    EMBEDDERS['potion']=dict(kind='static',ctor=MF,methods=['naive'],dims=[None],chunk='default')
    EMBEDDERS['jina']=dict(kind='late',ctor=ML,methods=['naive','latechunk'],dims=[None,16],chunk='long')
    EMBEDDERS['colbert']=dict(kind='colbert',ctor=MC,methods=['colbert'],dims=[None],chunk='small')
    docs={'d_eiffel':'the eiffel tower stands in paris france built for the world fair '*30,
          'd_cat':'cats are domestic mammals that purr and hunt mice as pets '*30,
          'd_py':'python is a programming language with dynamic typing and garbage collection '*30,
          'd_sea':'the ocean covers most of the earth surface and holds marine life '*30,
          'd_jazz':'jazz music features improvisation swing and blue notes from new orleans '*30,
          'd_mars':'mars is the red planet with two moons phobos and deimos in the solar system '*30}
    qs={'q1':'where is the eiffel tower located','q2':'which mammals purr and hunt mice',
        'q3':'programming language with dynamic typing','q4':'what covers most of the earth surface',
        'q5':'music with improvisation and swing','q6':'the red planet with two moons'}
    rel={'q1':{'d_eiffel'},'q2':{'d_cat'},'q3':{'d_py'},'q4':{'d_sea'},'q5':{'d_jazz'},'q6':{'d_mars'}}
    loader=lambda task,md,mq:(dict(docs),dict(qs),{k:set(v) for k,v in rel.items()})
    try:
        bm=Benchmark(tasks=['t'],max_docs=6,max_queries=6,cache=cache,loader=loader,sem_search=False)
        df=bm.run(expand_runs(models=['potion','jina','colbert'],add_rerank=False))
        # plumbing invariants (robust to toy-corpus noise): every route runs, ranks meaningfully, materializes stores
        assert set(df['recall@10'])=={1.0}                        # every route returns the right doc within top-10
        assert (df['ndcg@10']>0.6).all()                          # ranking is meaningful, not random
        assert {'jina-latechunk','jina-16-latechunk','jina-16-naive','colbert-colbert'} <= set(df.index)
        assert int(df.loc['colbert-colbert','chunks'])>0          # multivector store populated
        print('smoke test OK'); return df
    finally:
        EMBEDDERS.clear(); EMBEDDERS.update(reg)

smoke_test()